### 注意力机制与Transformers
注意力机制（Attention）是一个作用于向量集合的新原语（A new primitive）；Transformer：一个处处使用注意力机制的神经网络架构<br>
Transformer可以认为是RNN的一个分支，下图展示了RNN在序列到序列的应用
<div align="center">
  <img src="class_images/seq2seq_RNN.jpg" width="500">
</div>

上述算法的一个问题是encoder和decoder之间靠上下文向量（Context vector）来传递信息，但这个向量大小是固定的，因此当输入过长时，$c$可能无法储存全部信息。一个解决办法是去掉这个$c$，转而让神经网络在每生成一个输出单词时都能回看所有输入，这就是注意力机制与Transformers要解决的问题
#### 注意力机制（Attention）
<div align="center">
  <img src="class_images/seq2seq_RNN_Attention.jpg" width="500">
  <img src="class_images/seq2seq_RNN_Attention_following_step.jpg" width="500">
</div>

* 不同于一般的RNN，添加注意力机制后，我们首先从encoder的最后隐藏状态得到decoder的初始状态$s_0$，这或许是由最后隐藏状态通过线性变换得到的，或许直接初始化为零；然后利用$e_{t,i} = f_{att}(s_{t-1},h_i)$计算上一时刻隐藏状态（时刻$t$）与encoder每个隐藏状态的对齐分数，并通过softmax转化为概率$a_{t,i}$；接下来计算当前时刻$t$的上下文向量$c_t = \sum_{i}a_{t,i}h_i$；最后，decoder部分变化不大，重点在于通过$s_t = g_U(y_{t-1},s_{t-1},c_t)$得到当前时刻decoder的隐藏状态，然后得到输出$y_t$；重复以上过程就是利用带注意力机制的RNN进行seq2seq。简单来说，Attention就是通过引入每一步新计算的$c_t$来回看整个输入序列，关注哪些输入更相关来解决普通RNN的问题
* 虽然训练集既包含样本，也包含标签，例如翻译任务中给出原句，给出正确的目标句子，但单词之间的对应关系是算法无监督学习到的
* RNN seq2seq 里的 attention 可以被抽象成一个通用操作：每个 decoder state 作为 query，去关注所有 encoder states 这些 data vectors，并产生一个 context vector。这个抽象操作就是后面 Transformer 里 attention 机制的雏形
<div align="center">
  <img src="class_images/attention.jpg" width="500">
  <img src="class_images/self_attention.jpg" width="500">
</div>

* 左图展示了一般的注意力机制：首先对于输入$X$，我们通过$W_K$和$W_V$将它分为Key Matrix和Value Matrix，前者用于匹配，后者用于生成输出向量，例如对于一段话，Key Matrix可能包含关键词、索引等，使匹配更高效，而Value Matrix可能包含正文内容；然后使用内积计算$Q$和$K$的相似度，注意要除以$Q$的维数$\sqrt{D_Q}$，这是为了防止点积过大时，softmax 的输出会变得非常极端，接近 one-hot，导致大部分位置的梯度很小，训练不稳定；得到的相似度矩阵按行进行softmax（即dim=1，每一行相加等于1，与图中稍有不同）；最后用得到的概率加权得到输出$Y$
* 左图中，我们既有$X$，又有Query $Q$，这又称为交叉注意力（Cross Attention）。另一种常用的注意力机制是自注意力机制（右图），它的主要区别在于不再有$Q$，而是通过$W_Q$将$X$进一步分为$Q$，实践中，可以将$W_K$、$W_V$和$W_Q$组合成一个大矩阵进行计算
* 注意力机制与输入$X$中每个$X_i$的顺序无关，因此可以认为注意力机制并不是作用在序列上的，而是作用在一个矩阵集合上
<div align="center">
  <img src="class_images/multiheaded_self_attention.jpg" width="500">
</div>

上面展示了多头自注意力机制，它的好处是不同“头”可以关注上下文中的不同特征，例如一个关注主谓宾，一个关注时态，最后将不同头得到的输出按特征拼接在一起，并乘以矩阵$W_O$将维度变回输入维度，得到最终输出
#### Transformer
<div align="center">
  <img src="class_images/transformer.jpg" width="500">
  <img src="class_images/ViT.jpg" width="500">
</div>

* 从左图中可以看出transformer名字的由来：接到输入后，通过自注意力机制层、规范化层和MLP层将四个输入转化为四个输出，其中有且仅有自注意力层考虑到了各个输入向量之间的关系
* 视觉Transformer（ViT）：ViT 把图片切成固定大小的 patch，把每个 patch 展平并映射成一个向量，再把这些向量当成 Transformer 的输入 token；Transformer 输出每个 patch 的表示，最后通过 pooling 和线性分类器得到整张图片的类别预测